## Question 3: Fine-Tuning a Detection Model
Using TensorFlow or PyTorch, load a pre-trained model (e.g., ResNet50, EfficientNet, MobileNet) and fine-tune it on KITTI. Your tasks include:

- Load the pre-trained model.
- Split the train set (train 70%, validation 15%, test 15%)
- Modify the model's architecture to reflect the number of classes of KITTI.
- Fine-Tune based using any strategy you want.
- Report the model's performance on the test set.

In [ ]:
import torchvision
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.transforms import functional as F
from torchvision.datasets import Kitti
from torch.utils.data import DataLoader
from torch.optim import SGD
import torch

import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
def visualize_boxes(img, bounding_boxes, annotations):
  """
  Helper function to visualize image with all of it's bounding boxes and annotations.
  """

  fig, ax = plt.subplots(1)

  ax.imshow(img)

  for bbox, annotation in zip(bounding_boxes, annotations):

      x_min, y_min, x_max, y_max = bbox
      width = x_max - x_min
      height = y_max - y_min

      rect = patches.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor='r', facecolor='none')

      ax.add_patch(rect)

      plt.text(x_min, y_min - 5, annotation, color='r', fontsize=8, bbox=dict(facecolor='white', alpha=0.7))

  plt.axis('off')
  plt.show()

def custom_collate_fn(data):
  return tuple(zip(*data))

In [ ]:
# Define the dataset path and transform
data_path = "./train/"
transform = torchvision.transforms.Compose([
    torchvision.transforms.ToTensor(),
    # hint: there is something missin here
    ])

train_dataset = Kitti(
    data_path,
    train= True, # hint: its a boolean
    transform=transform,
    download=True
    )

# Load a pre-trained model

In [ ]:
kiti_classes = {'Car': 0,
 'Cyclist': 1,
 'DontCare': 2,
 'Misc': 3,
 'Pedestrian': 4,
 'Person_sitting': 5,
 'Tram': 6,
 'Truck': 7,
 'Van': 8}

In [ ]:
kiti_voc_classes = {v: k for k, v in kiti_classes.items()}

In [ ]:
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained=True)

# Split the train set (Train 70%, Validation 15%, Test 15%)


In [ ]:
# split the data here
generator = torch.Generator().manual_seed(42)
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(train_dataset, [0.7, 0.15, 0.15], generator=generator)


# Create data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=custom_collate_fn)
valid_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, collate_fn=custom_collate_fn)

# Modify the model's architecture to reflect the number of classes of KITTI.

In [ ]:
num_classes = 9 # Number of classes for KITTI dataset

in_features = model.roi_heads.box_predictor.cls_score.in_features  # input features of bounding box predictor
prev_num_classes = model.roi_heads.box_predictor.cls_score.out_features

# update the bounding box prediction layers with new untrained layers which now predict 20 classes instead of previous.
model.roi_heads.box_predictor = torchvision.models.detection.faster_rcnn.FastRCNNPredictor(in_features, num_classes)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

print('Old Num Classes: ', prev_num_classes)
print('Using device', device)

In [ ]:
# No need to train the backbone
# We CAN but it'll take too long and too much computation

model.requires_grad_(False)
model.roi_heads.box_predictor = model.roi_heads.box_predictor.requires_grad_(True)

In [ ]:
params = [p for p in model.parameters() if p.requires_grad]
# optimizer = SGD(params, lr=1e-5)
optimizer = SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

num_epochs = 10

# Fine-Tune based using any strategy you want.

In [ ]:
from tqdm import tqdm
for epoch in range(num_epochs):
    model.train()
    for images, targets in tqdm(train_loader):

        images = list(image.to(device) for image in images)
        new_targets = []
        for target in targets:
          boxes = []
          labels = []
          for obj in target:
            label = obj['type']
            box = obj['bbox']
            # xmin, ymin, xmax, ymax = [int(box[k]) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
            box = torch.Tensor(box).to(device)
            boxes.append(box)
            labels.append(kiti_classes[label])
          target = {}
          target['boxes'] = torch.stack(boxes)
          target['labels'] = torch.Tensor(labels).type(torch.int64).to(device)
          new_targets.append(target)


        # In training mode, the model will take images and targets and calculate all the losses in internal implementation
        loss_dict = model(images, new_targets)
        losses = sum(loss for loss in loss_dict.values())  # Sum all the different losses.

        optimizer.zero_grad()
        losses.backward()
        optimizer.step()

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {losses.item()}")

# Report the model's performance on the test set.

In [ ]:
model.eval()

In [ ]:
test_img, test_target = test_dataset[10]

test_boxes = []
annotations = []
for obj in test_target:
  box = obj['bbox']
  # xmin, ymin, xmax, ymax = [int(box[k]) for k in ['xmin', 'ymin', 'xmax', 'ymax']]
  # box = [xmin, ymin, xmax, ymax]
  test_boxes.append(box)
  annotations.append(obj['type'])

with torch.no_grad():
  pred = model([test_img.to(device)])

pred = pred[0]

pred_boxes = pred['boxes']
pred_annotations = pred['labels']
pred_scores = pred['scores']

valid_mask = pred_scores>=0.3

pred_annotations = pred_annotations[valid_mask]
pred_boxes = pred_boxes[valid_mask]

pred_annotations = [kiti_voc_classes[val.item()] for val in pred_annotations]

In [ ]:
# Ground truth
visualize_boxes(F.to_pil_image(test_img), test_boxes, annotations)

In [ ]:
# Predicted
visualize_boxes(F.to_pil_image(test_img), pred_boxes.cpu(), pred_annotations)

In [ ]:
pred_boxes

In [ ]:
pred